# 0.Instalación de librería necesarias

In [25]:
#Ejecutar cada vez que se abra el notebook
# PD: Este noteebok se ha trabajado creando un kernel con las librerias instaladas.

%pip install pandas numpy matplotlib seaborn plotly mysql-connector-python sqlalchemy openpyxl xlrd


# 1. Función para transformación de los JSON en DF.

In [26]:
# ============================================================
# CARGA DE ARCHIVOS DEL PROYECTO EN GOOGLE COLAB
# ============================================================

from google.colab import files
from pathlib import Path


# ============================================================
# 1. RUTAS DE TRABAJO
# ============================================================

ruta_json = Path("/content")
ruta_subtipos = Path("/content/subtipos.csv")

print("Ruta de trabajo para los JSON:", ruta_json)
print("Ruta de subtipos:", ruta_subtipos)


# ============================================================
# 2. ARCHIVOS JSON NECESARIOS
# ============================================================

archivos_json = {
    "lotes.json",
    "info_lote.json",
    "tipo.json",
    "proveedores.json",
    "proveedores-clientes.json",
    "venta_en_cliente.json"
}


# ============================================================
# 3. COMPROBAR QUÉ JSON YA ESTÁN CARGADOS
# ============================================================

existentes = {
    archivo.name
    for archivo in ruta_json.iterdir()
    if archivo.is_file()
}

faltantes = archivos_json - existentes


# ============================================================
# 4. SUBIR LOS JSON SI FALTAN
# ============================================================

if faltantes:

    print("\nFaltan estos JSON:")

    for nombre in sorted(faltantes):
        print(f" - {nombre}")

    print(
        "\nSelecciona los JSON desde tu ordenador.\n"
        "Puedes seleccionar los 6 a la vez."
    )

    uploaded = files.upload()

    # Guardamos explícitamente los archivos en /content
    for nombre, contenido in uploaded.items():
        (ruta_json / nombre).write_bytes(contenido)


# ============================================================
# 5. VERIFICAR LOS JSON
# ============================================================

existentes = {
    archivo.name
    for archivo in ruta_json.iterdir()
    if archivo.is_file()
}

faltantes = archivos_json - existentes

if faltantes:

    raise FileNotFoundError(
        "Siguen faltando estos archivos JSON:\n"
        + "\n".join(
            f" - {nombre}"
            for nombre in sorted(faltantes)
        )
    )

print("\nJSON disponibles correctamente.")

for nombre in sorted(archivos_json):
    print(f" ✓ {nombre}")


# ============================================================
# 6. COMPROBAR SI subtipos.csv YA ESTÁ CARGADO
# ============================================================

if ruta_subtipos.exists():

    print("\nsubtipos.csv ya está disponible.")
    print(f" ✓ {ruta_subtipos}")

else:

    # ========================================================
    # 7. SUBIR subtipos.csv
    # ========================================================

    print(
        "\nAhora selecciona el archivo subtipos.csv "
        "generado a partir de la carpeta archive."
    )

    uploaded_subtipos = files.upload()

    encontrado = False

    for nombre, contenido in uploaded_subtipos.items():

        if nombre.lower() == "subtipos.csv":

            ruta_subtipos.write_bytes(contenido)
            encontrado = True
            break


    if not encontrado:

        raise FileNotFoundError(
            "No se ha subido un archivo llamado subtipos.csv."
        )


# ============================================================
# 8. COMPROBAR subtipos.csv
# ============================================================

if not ruta_subtipos.exists():

    raise FileNotFoundError(
        f"No se ha encontrado:\n{ruta_subtipos}"
    )


print("\nsubtipos.csv disponible correctamente.")
print(f" ✓ {ruta_subtipos.name}")


# ============================================================
# 9. RESUMEN FINAL
# ============================================================

print("\n-----------------------------------")
print("CONFIGURACIÓN COMPLETADA")
print("-----------------------------------")
print(f"JSON:      {ruta_json}")
print(f"Subtipos:  {ruta_subtipos}")

Ruta de trabajo para los JSON: /content
Ruta de subtipos: /content/subtipos.csv

JSON disponibles correctamente.
 ✓ info_lote.json
 ✓ lotes.json
 ✓ proveedores-clientes.json
 ✓ proveedores.json
 ✓ tipo.json
 ✓ venta_en_cliente.json

Ahora selecciona el archivo subtipos.csv generado a partir de la carpeta archive.


Saving subtipos.csv to subtipos.csv

subtipos.csv disponible correctamente.
 ✓ subtipos.csv

-----------------------------------
CONFIGURACIÓN COMPLETADA
-----------------------------------
JSON:      /content
Subtipos:  /content/subtipos.csv


In [27]:
#Función para convertir los JSON en Dataframes
import json
from pathlib import Path
import pandas as pd

# Configuración de pandas para ver bien los DataFrames
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

def cargar_jsons(ruta_carpeta):
    ruta = Path(ruta_carpeta)
    def _leer(nombre):
        with open(ruta / nombre, "r", encoding="utf-8") as f:
            return json.load(f)

    raw = _leer("lotes.json")
    df_lotes = pd.DataFrame(
        list(raw.items()),
        columns=["t_id", "codigo_lote"]
    )
    raw = _leer("info_lote.json")
    df_info_lote = pd.concat(
        [pd.DataFrame(lista).assign(codigo_lote=lote)
         for lote, lista in raw.items()],
        ignore_index=True
    )
    raw = _leer("tipo.json")
    df_tipo = pd.concat(
        [pd.DataFrame(lista) for lista in raw.values()],
        ignore_index=True
    )
    raw = _leer("proveedores.json")
    df_proveedores = pd.concat(
        [pd.DataFrame(lista) for lista in raw.values()],
        ignore_index=True
    )
    raw = _leer("proveedores-clientes.json")
    df_prov_cli = pd.DataFrame([
        {"proveedor": prov, "t_id": t_id, "cliente": cli}
        for prov, pares in raw.items()
        for t_id, cli in pares
    ])

    raw = _leer("venta_en_cliente.json")
    df_venta = pd.concat(
        [pd.DataFrame(lista).assign(cliente=cli)
         for cli, lista in raw.items()],
        ignore_index=True
    )
    return {
        "lotes": df_lotes,
        "info_lote": df_info_lote,
        "tipo": df_tipo,
        "proveedores_clientes": df_prov_cli,
        "venta": df_venta,
        "proveedores":df_proveedores
    }

# Creamos un diccionario con todos los data frames y revisamos que estén correctos:
datos = cargar_jsons(ruta_json)

/tmp/ipykernel_1591/48608404.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_info_lote = pd.concat(


In [28]:
for nombre, df in datos.items():
    print(f"{nombre}: {df.shape}")

lotes: (69687, 2)
info_lote: (70549, 4)
tipo: (70549, 3)
proveedores_clientes: (70549, 3)
venta: (70549, 5)
proveedores: (70549, 2)


In [29]:
# Diagnóstico inicial: duplicados y consistencia de t_id

# 1. ¿Hay duplicados de t_id dentro de cada tabla?
print("=== Duplicados de t_id en cada tabla ===")
for nombre, df in datos.items():
    if "t_id" in df.columns:
        n_dup = df["t_id"].duplicated().sum()
        n_unicos = df["t_id"].nunique()
        print(f"{nombre:25s} filas={len(df):>6}  t_id únicos={n_unicos:>6}  duplicados={n_dup}")

# 2. Conjunto de t_id por tabla
print("\n=== t_id presentes en cada tabla (set) ===")
sets_tid = {nombre: set(df["t_id"]) for nombre, df in datos.items() if "t_id" in df.columns}
for nombre, s in sets_tid.items():
    print(f"{nombre:25s} {len(s)} t_id únicos")

# 3. ¿Qué t_id están en info_lote pero no en lotes?
faltan_en_lotes = sets_tid["info_lote"] - sets_tid["lotes"]
print(f"\nt_id en 'info_lote' pero NO en 'lotes': {len(faltan_en_lotes)}")
print("Ejemplos (primeros 15):")
for t in list(faltan_en_lotes)[:15]:
    print(f"  - {t}")

=== Duplicados de t_id en cada tabla ===
lotes                     filas= 69687  t_id únicos= 69687  duplicados=0
info_lote                 filas= 70549  t_id únicos= 69687  duplicados=862
tipo                      filas= 70549  t_id únicos= 69687  duplicados=862
proveedores_clientes      filas= 70549  t_id únicos= 69687  duplicados=862
venta                     filas= 70549  t_id únicos= 69687  duplicados=862
proveedores               filas= 70549  t_id únicos= 69687  duplicados=862

=== t_id presentes en cada tabla (set) ===
lotes                     69687 t_id únicos
info_lote                 69687 t_id únicos
tipo                      69687 t_id únicos
proveedores_clientes      69687 t_id únicos
venta                     69687 t_id únicos
proveedores               69687 t_id únicos

t_id en 'info_lote' pero NO en 'lotes': 0
Ejemplos (primeros 15):


In [30]:
# Analisis de los 862 t_id duplicados

# Cojo los t_id que están duplicados en info_lote (deberían ser los mismos en las otras 4)
t_id_duplicados = datos["info_lote"][datos["info_lote"]["t_id"].duplicated(keep=False)]["t_id"].unique()
print(f"t_id duplicados: {len(t_id_duplicados)}")

# Cogemos uno de ejemplo:
ejemplo = t_id_duplicados[0]
print(f"\n=== Ejemplo: '{ejemplo}' ===\n")

for nombre, df in datos.items():
    if "t_id" in df.columns:
        filas = df[df["t_id"] == ejemplo]
        print(f"--- {nombre} ({len(filas)} filas) ---")
        print(filas.to_string(index=False))
        print()

t_id duplicados: 597

=== Ejemplo: 'scene00201.png' ===

--- lotes (1 filas) ---
          t_id           codigo_lote
scene00201.png C67K78K48L50L49J80T71

--- info_lote (4 filas) ---
          t_id            marca  coste_inicial           codigo_lote
scene00201.png  VibranteSabores       1.474485 C67K78K48L50L49J80T71
scene00201.png    DeliciosoEdén       1.924915 C67K78K48L50L49J80T71
scene00201.png    PaladarDorado       2.688805 C67K78K48L50L49J80T71
scene00201.png DulzuraSilvestre       2.420529 C67K78K48L50L49J80T71

--- tipo (4 filas) ---
          t_id  tipo  tiempo_recogida
scene00201.png Apple              175
scene00201.png Apple              346
scene00201.png Apple              485
scene00201.png Guava              647

--- proveedores_clientes (4 filas) ---
                           proveedor           t_id                cliente
Agricultura Inteligente TechCultivos scene00201.png     Alimentación Total
        Carnes Sostenibles CampoReal scene00201.png Distribuciones 

In [31]:
# 1. ¿Todos los duplicados empiezan por "scene"?
import pandas as pd

df_dup = datos["info_lote"][datos["info_lote"]["t_id"].duplicated(keep=False)]
empiezan_scene = df_dup["t_id"].str.startswith("scene").sum()
total_filas_dup = len(df_dup)
print(f"Filas duplicadas: {total_filas_dup}")
print(f"De ellas que empiezan por 'scene': {empiezan_scene}")
print(f"Que NO empiezan por 'scene': {total_filas_dup - empiezan_scene}")

# 2. Cuántos scene en total vs cuántos t_id únicos duplicados
t_id_dup_unicos = df_dup["t_id"].unique()
scene_unicos = sum(1 for t in t_id_dup_unicos if t.startswith("scene"))
print(f"\nt_id únicos duplicados: {len(t_id_dup_unicos)}")
print(f"De ellos que empiezan por 'scene': {scene_unicos}")

# 3. ¿Los t_id "buenos" (Apple X.png) también tienen duplicados?
no_scene_dup = [t for t in t_id_dup_unicos if not t.startswith("scene")]
print(f"\nt_id duplicados que NO son 'scene' ({len(no_scene_dup)} ejemplos):")
for t in no_scene_dup[:20]:
    print(f"  - {t}")

Filas duplicadas: 1459
De ellas que empiezan por 'scene': 1459
Que NO empiezan por 'scene': 0

t_id únicos duplicados: 597
De ellos que empiezan por 'scene': 597

t_id duplicados que NO son 'scene' (0 ejemplos):


## 1.1. Extracción de los subtipos
Función que recorre la carpeta 'archive' donde se encuentran las imagenes de cada entrada de fruta dividas en tipo y subtipos.

In [32]:
# ============================================================
# CARGAR LA COMPOSICIÓN DE archive DESDE subtipos.csv
# ============================================================

import pandas as pd

df_subtipos = pd.read_csv(
    ruta_subtipos,
    encoding="utf-8-sig"
)


# Comprobamos que el CSV tiene las columnas esperadas
columnas_esperadas = {"t_id", "tipo", "subtipo"}

faltantes = columnas_esperadas - set(df_subtipos.columns)

if faltantes:

    raise ValueError(
        "Faltan columnas en subtipos.csv:\n"
        + "\n".join(
            f" - {columna}"
            for columna in sorted(faltantes)
        )
    )


print(f"Imágenes encontradas: {len(df_subtipos)}")
print(f"Tipos distintos: {df_subtipos['tipo'].nunique()}")
print(f"Subtipos distintos: {df_subtipos['subtipo'].nunique()}")

df_subtipos.head(10)

Imágenes encontradas: 70549
Tipos distintos: 15
Subtipos distintos: 27


,t_id,subtipo,tipo
0,Apple 1.png,Apple A,Apple
1,Apple 10.png,Apple A,Apple
2,Apple 100.png,Apple A,Apple
3,Apple 101.png,Apple A,Apple
4,Apple 102.png,Apple A,Apple
5,Apple 103.png,Apple A,Apple
6,Apple 104.png,Apple A,Apple
7,Apple 105.png,Apple A,Apple
8,Apple 106.png,Apple A,Apple
9,Apple 107.png,Apple A,Apple


In [33]:
# t_id que están en los JSON pero NO existen como imagen en disco
en_json_sin_imagen = set(datos["tipo"]["t_id"]) - set(df_subtipos["t_id"])

# t_id que existen como imagen pero NO aparecen en los JSON
en_imagen_sin_json = set(df_subtipos["t_id"]) - set(datos["tipo"]["t_id"])

print(f"En JSON pero sin imagen: {len(en_json_sin_imagen)}")
print(f"En imagen pero sin JSON: {len(en_imagen_sin_json)}")

df_subtipos.groupby("tipo")["subtipo"].nunique().sort_values(ascending=False)

En JSON pero sin imagen: 0
En imagen pero sin JSON: 0


,subtipo
tipo,
Apple,7
Kiwi,4
Guava,4
Carambola,1
Banana,1
Mango,1
Orange,1
Peach,1
Pear,1


In [34]:
# ¿Cuántos t_id únicos hay en df_subtipos?
print(f"Filas en df_subtipos: {len(df_subtipos)}")
print(f"t_id únicos: {df_subtipos['t_id'].nunique()}")
print(f"t_id duplicados: {df_subtipos['t_id'].duplicated().sum()}")

# Si hay duplicados, ¿en qué subtipos aparecen los t_id repetidos?
duplicados = df_subtipos[df_subtipos["t_id"].duplicated(keep=False)]
print(f"\nFilas implicadas en duplicación: {len(duplicados)}")
print(f"\nSubtipos donde aparecen t_id duplicados:")
print(duplicados["subtipo"].value_counts())

Filas en df_subtipos: 70549
t_id únicos: 69687
t_id duplicados: 862

Filas implicadas en duplicación: 1459

Subtipos donde aparecen t_id duplicados:
subtipo
guava A    512
Apple C    482
Apple F    285
Apple B    101
Apple D     79
Name: count, dtype: int64


In [35]:
# t_id duplicados en disco
dup_disco = set(df_subtipos[df_subtipos["t_id"].duplicated(keep=False)]["t_id"])

# t_id duplicados en JSON (info_lote)
dup_json = set(datos["info_lote"][datos["info_lote"]["t_id"].duplicated(keep=False)]["t_id"])

print(f"t_id duplicados en disco: {len(dup_disco)}")
print(f"t_id duplicados en JSON:  {len(dup_json)}")
print(f"Coincidencia exacta:      {dup_disco == dup_json}")
print(f"En disco pero no en JSON: {len(dup_disco - dup_json)}")
print(f"En JSON pero no en disco: {len(dup_json - dup_disco)}")

t_id duplicados en disco: 597
t_id duplicados en JSON:  597
Coincidencia exacta:      True
En disco pero no en JSON: 0
En JSON pero no en disco: 0


In [36]:
# ¿Qué hay en las "carpetas de totales"?
sospechosas = df_subtipos[df_subtipos["subtipo"].str.contains("total|Total|final", case=False)]
print(f"Imágenes en carpetas con 'total' o 'final' en el nombre: {len(sospechosas)}")
print(sospechosas["subtipo"].value_counts())

Imágenes en carpetas con 'total' o 'final' en el nombre: 25757
subtipo
Guava total                   12552
Total Number of Apples         5024
Total Number of Kiwi fruit     4173
guava total final              4008
Name: count, dtype: int64


In [37]:
# Para cada carpeta "total/final", cuántos de sus t_id están solo ahí vs aparecen también en otras carpetas
sospechosas = df_subtipos[df_subtipos["subtipo"].str.contains("total|final", case=False)]
t_id_en_sospechosas = set(sospechosas["t_id"])
t_id_en_otras = set(df_subtipos[~df_subtipos["subtipo"].str.contains("total|final", case=False)]["t_id"])

solo_en_sospechosas = t_id_en_sospechosas - t_id_en_otras
en_ambas = t_id_en_sospechosas & t_id_en_otras

print(f"t_id en carpetas 'total/final': {len(t_id_en_sospechosas)}")
print(f"  - Solo en esas carpetas: {len(solo_en_sospechosas)}")
print(f"  - También en otras subcarpetas: {len(en_ambas)}")

t_id en carpetas 'total/final': 25757
  - Solo en esas carpetas: 25757
  - También en otras subcarpetas: 0


Se ve que los t_id coinciden con los t_id de los JSON y por lo tanto podemos enriquecer con los datos de subtipos de frutas el modelo. Los subtipos denominados como "Total number of..." no son agregadores, contienen t_id únicos. Por lo tanto, se presupone que estos subtipos se han creado para frutas no categorizables en el resto de subtipos y, se pueden usar como subtipos reales. Posteriormente en la limpieza de datos adaptaremos su nombre.

# 2. Creación de los DF para carga en SQL

## 2.1. Análisis t_id duplicados

In [38]:
# Para un scene de prueba, vemos si las apariciones tienen un orden interpretable
ejemplo = "scene00201.png"

for nombre in ["info_lote", "tipo", "proveedores", "proveedores_clientes", "venta"]:
    df = datos[nombre]
    filas = df[df["t_id"] == ejemplo].reset_index(drop=True)
    print(f"\n=== {nombre} ===")
    print(filas)


=== info_lote ===
             t_id             marca  coste_inicial            codigo_lote
0  scene00201.png   VibranteSabores       1.474485  C67K78K48L50L49J80T71
1  scene00201.png     DeliciosoEdén       1.924915  C67K78K48L50L49J80T71
2  scene00201.png     PaladarDorado       2.688805  C67K78K48L50L49J80T71
3  scene00201.png  DulzuraSilvestre       2.420529  C67K78K48L50L49J80T71

=== tipo ===
             t_id   tipo  tiempo_recogida
0  scene00201.png  Apple              175
1  scene00201.png  Apple              346
2  scene00201.png  Apple              485
3  scene00201.png  Guava              647

=== proveedores ===
             t_id                             proveedor
0  scene00201.png  Agricultura Inteligente TechCultivos
1  scene00201.png          Carnes Sostenibles CampoReal
2  scene00201.png                 Veterinaria EcoAnimal
3  scene00201.png              Agrícola Solaris Energía

=== proveedores_clientes ===
                              proveedor            t_id 

Se decide hacer una asunción de valores en base al orden de aparcición de los duplicados. Tras el analisis se puede observar que los t_id duplicados no son errores de productos duplicados, si no un error en el etiquetado. Por lo tanto, se presupone que se han ido generando en orden según se han ido etiquetando.

In [39]:
ejemplo = "scene00201.png"
print("=== df_subtipos ===")
print(df_subtipos[df_subtipos["t_id"] == ejemplo].reset_index(drop=True))

=== df_subtipos ===
             t_id  subtipo   tipo
0  scene00201.png  Apple B  Apple
1  scene00201.png  Apple D  Apple
2  scene00201.png  Apple F  Apple
3  scene00201.png  guava A  Guava


In [40]:
# Para los t_id duplicados, ¿hay alguno con proveedor repetido dentro del t_id?
dup_provs = datos["proveedores"][datos["proveedores"]["t_id"].isin(
    datos["proveedores"][datos["proveedores"]["t_id"].duplicated(keep=False)]["t_id"]
)]
prov_repetido = dup_provs.groupby("t_id")["proveedor"].apply(lambda x: x.duplicated().any()).sum()
print(f"t_id con proveedor repetido en sus apariciones: {prov_repetido}")

# Lo mismo para cliente en venta
dup_ventas = datos["venta"][datos["venta"]["t_id"].isin(
    datos["venta"][datos["venta"]["t_id"].duplicated(keep=False)]["t_id"]
)]
cli_repetido = dup_ventas.groupby("t_id")["cliente"].apply(lambda x: x.duplicated().any()).sum()
print(f"t_id con cliente repetido en sus apariciones: {cli_repetido}")

t_id con proveedor repetido en sus apariciones: 25
t_id con cliente repetido en sus apariciones: 37


In [41]:
dup_subtipos = df_subtipos[df_subtipos["t_id"].duplicated(keep=False)]
sub_repetido = dup_subtipos.groupby("t_id")["subtipo"].apply(lambda x: x.duplicated().any()).sum()
print(f"t_id con subtipo repetido en sus apariciones: {sub_repetido}")

t_id con subtipo repetido en sus apariciones: 0


## 2.2. Construcción de los DF

### 1º) Función para crear un diccionario con todos los DF.

def construccion_df_final(datos, df_subtipos):
    df = datos["info_lote"].copy()
    df["n_aparicion"] = df.groupby("t_id").cumcount()

    df_tipo = datos["tipo"].copy()
    df_tipo["n_aparicion"] = df_tipo.groupby("t_id").cumcount()
    df = df.merge(df_tipo, on=["t_id", "n_aparicion"], how="left")

    df_prov = datos["proveedores"].copy()
    df_prov["n_aparicion"] = df_prov.groupby("t_id").cumcount()
    df = df.merge(df_prov, on=["t_id", "n_aparicion"], how="left")

    df_pc = datos["proveedores_clientes"].copy()
    df_pc["n_aparicion"] = df_pc.groupby("t_id").cumcount()

    df_pc = df_pc.drop(columns=["proveedor"])
    df = df.merge(df_pc, on=["t_id", "n_aparicion"], how="left")

    df_venta = datos["venta"].copy()
    df_venta["n_aparicion"] = df_venta.groupby("t_id").cumcount()

    df_venta = df_venta.drop(columns=["cliente"])
    df = df.merge(df_venta, on=["t_id", "n_aparicion"], how="left")

    df["n_aparicion_tipo"] = df.groupby(["t_id", "tipo"]).cumcount()

    df_sub = df_subtipos.copy()
    df_sub["n_aparicion_tipo"] = df_sub.groupby(["t_id", "tipo"]).cumcount()
    df = df.merge(df_sub, on=["t_id", "tipo", "n_aparicion_tipo"], how="left")

    df_lotes = datos["lotes"].copy().rename(columns={"codigo_lote": "codigo_lote_check"})
    df = df.merge(df_lotes, on="t_id", how="left")

    inconsistencias = (df["codigo_lote"] != df["codigo_lote_check"]).sum()
    if inconsistencias > 0:
        print(f"{inconsistencias} filas con codigo_lote distinto entre info_lote y lotes.json")
    else:
        print("codigo_lote consistente entre info_lote y lotes.json")

    df = df.drop(columns=["codigo_lote_check"])

    df = df.drop(columns=["n_aparicion", "n_aparicion_tipo"])

    columnas_orden = [
        "t_id", "tipo", "subtipo", "marca", "codigo_lote",
        "proveedor", "cliente",
        "coste_inicial", "tiempo_recogida",
        "precio_venta", "tiempo_venta", "peso",
    ]
    df = df[columnas_orden]

    return df


In [43]:
def construccion_df_final(datos, df_subtipos):
    """
    Construye el DataFrame base que posteriormente se transforma
    en las tablas relacionales de SQL.

    IMPORTANTE
    ----------
    - No modifica ningún valor de los JSON.
    - No elimina nulos.
    - No corrige tiempos, precios, pesos ni costes.
    - No infiere relaciones a partir de valores económicos o temporales.
    - Mantiene exactamente las columnas que espera
      construccion_tablas_relacionales().

    HIPÓTESIS DE TRABAJO
    -------------------
    Cuando un mismo t_id aparece varias veces, se asume que la posición
    relativa de sus apariciones representa la correspondencia entre
    los distintos ficheros:

        primera aparición  <-> primera aparición
        segunda aparición  <-> segunda aparición
        ...

    La posición se utiliza únicamente como clave técnica temporal.
    No formará parte de las tablas SQL finales.
    """


    # 1. COPIAS DE TRABAJO

    # Trabajamos sobre copias para no modificar los DataFrames
    # originales almacenados en "datos".


    df_info = datos["info_lote"].copy().reset_index(drop=True)
    df_tipo = datos["tipo"].copy().reset_index(drop=True)
    df_prov = datos["proveedores"].copy().reset_index(drop=True)
    df_pc = datos["proveedores_clientes"].copy().reset_index(drop=True)
    df_venta = datos["venta"].copy().reset_index(drop=True)
    df_lotes = datos["lotes"].copy().reset_index(drop=True)

    df_sub = df_subtipos.copy().reset_index(drop=True)



    # 2. COMPROBACIÓN DE MULTIPLICIDAD DE t_id

    # Todas estas tablas describen las mismas 70.549 apariciones
    # de producto.
    #
    # Por tanto, cada t_id debe aparecer el mismo número de veces
    # en:
    #
    # - info_lote
    # - tipo
    # - proveedores
    # - proveedores-clientes
    # - venta_en_cliente
    # - df_subtipos
    #
    # Esto no obliga a que estén en el mismo orden global.
    # Sólo comprueba que contienen las mismas ocurrencias.


    def conteo_tid(df):
        return (
            df.groupby("t_id", dropna=False)
              .size()
              .sort_index()
        )


    conteo_referencia = conteo_tid(df_info)

    tablas_a_comprobar = {
        "tipo": df_tipo,
        "proveedores": df_prov,
        "proveedores-clientes": df_pc,
        "venta_en_cliente": df_venta,
        "subtipos": df_sub,
    }


    for nombre, tabla in tablas_a_comprobar.items():

        if not conteo_tid(tabla).equals(conteo_referencia):

            raise ValueError(
                f"La multiplicidad de t_id en '{nombre}' "
                "no coincide con info_lote."
            )


    # Guardamos el número esperado de productos.
    # Al terminar debemos seguir teniendo exactamente estas filas.
    filas_esperadas = len(df_info)



    # 3. POSICIÓN DE CADA APARICIÓN DE t_id

    # Ejemplo:
    #
    # scene00201.png   -> posición 0
    # scene00201.png   -> posición 1
    # scene00201.png   -> posición 2
    # scene00201.png   -> posición 3
    #
    # Esta columna NO es un dato de negocio.
    #
    # Es simplemente la implementación de la hipótesis acordada:
    # si el t_id está repetido, la posición indica la relación.


    df_info["_pos_tid"] = (
        df_info
        .groupby("t_id", dropna=False)
        .cumcount()
    )

    df_tipo["_pos_tid"] = (
        df_tipo
        .groupby("t_id", dropna=False)
        .cumcount()
    )

    df_prov["_pos_tid"] = (
        df_prov
        .groupby("t_id", dropna=False)
        .cumcount()
    )

    df_pc["_pos_tid"] = (
        df_pc
        .groupby("t_id", dropna=False)
        .cumcount()
    )



    # 4. PROVEEDORES + PROVEEDORES-CLIENTES

    # proveedores-clientes contiene directamente:
    #
    #     proveedor
    #     t_id
    #     cliente
    #
    # Ésta será nuestra tabla puente.
    #
    # proveedores.json contiene:
    #
    #     t_id
    #     proveedor
    #
    # Como sabemos que Value.proveedor coincide con el Name del
    # proveedor, utilizamos proveedores.json para comprobar que
    # proveedor y posición son consistentes con proveedores-clientes.
    #
    # Para t_id repetidos:
    #
    #     t_id + _pos_tid
    #
    # identifica cada aparición según la hipótesis acordada.


    df_relacion = df_pc.merge(

        df_prov[
            [
                "t_id",
                "_pos_tid",
                "proveedor"
            ]
        ].rename(
            columns={
                "proveedor": "proveedor_check"
            }
        ),

        on=[
            "t_id",
            "_pos_tid"
        ],

        how="left",

        validate="one_to_one",

        indicator="_merge_proveedor"
    )



    # Comprobamos que todas las filas de proveedores-clientes
    # encuentran su correspondiente fila en proveedores.json.


    if not (
        df_relacion["_merge_proveedor"] == "both"
    ).all():

        raise ValueError(
            "Hay registros de proveedores-clientes que no "
            "encuentran correspondencia en proveedores.json."
        )



    # Comprobamos además que el proveedor coincide.
    #
    # No corregimos nada:
    # si hay discrepancia se detiene la construcción.


    proveedores_distintos = (
        df_relacion["proveedor"]
        != df_relacion["proveedor_check"]
    )


    if proveedores_distintos.any():

        raise ValueError(
            f"Se han encontrado "
            f"{proveedores_distintos.sum()} relaciones donde "
            "proveedores.json y proveedores-clientes.json "
            "asignan proveedores diferentes."
        )


    # Eliminamos las columnas utilizadas sólo para comprobación.
    df_relacion = df_relacion.drop(
        columns=[
            "proveedor_check",
            "_merge_proveedor"
        ]
    )



    # 5. PROVEEDORES-CLIENTES + VENTA_EN_CLIENTE

    # Esta relación NO se realiza únicamente por posición de t_id.
    #
    # Tenemos una relación explícita más fuerte:
    #
    #     t_id + cliente
    #
    # proveedores-clientes:
    #
    #     t_id
    #     cliente
    #     proveedor
    #
    # venta_en_cliente:
    #
    #     t_id
    #     cliente
    #     tiempo_venta
    #     precio_venta
    #     peso
    #
    # Por tanto usamos:
    #
    #     t_id + cliente
    #
    # para incorporar los datos de venta.
    #
    # Hay algunos casos donde incluso t_id + cliente se repite.
    # Esas filas representan transacciones independientes.
    #
    # Para evitar que merge() genere un producto cartesiano
    # 2 x 2 = 4 filas, numeramos sólo las repeticiones dentro
    # de cada pareja t_id + cliente.


    df_relacion["_pos_transaccion"] = (
        df_relacion
        .groupby(
            [
                "t_id",
                "cliente"
            ],
            dropna=False
        )
        .cumcount()
    )


    df_venta["_pos_transaccion"] = (
        df_venta
        .groupby(
            [
                "t_id",
                "cliente"
            ],
            dropna=False
        )
        .cumcount()
    )



    # 6. COMPROBAR QUE AMBAS TABLAS TIENEN EL MISMO NÚMERO
    #    DE TRANSACCIONES POR t_id + cliente

    #
    # Por ejemplo, si:
    #
    # proveedores-clientes:
    #
    #     X.png + Cliente A -> 2 registros
    #
    # entonces venta_en_cliente también debe tener:
    #
    #     X.png + Cliente A -> 2 registros
    #
    # No se estudia aquí si sus valores son correctos.
    # Sólo se comprueba la estructura de la relación.


    conteo_pc_cliente = (
        df_relacion
        .groupby(
            [
                "t_id",
                "cliente"
            ],
            dropna=False
        )
        .size()
        .sort_index()
    )


    conteo_venta_cliente = (
        df_venta
        .groupby(
            [
                "t_id",
                "cliente"
            ],
            dropna=False
        )
        .size()
        .sort_index()
    )


    if not conteo_pc_cliente.equals(
        conteo_venta_cliente
    ):

        raise ValueError(
            "proveedores-clientes y venta_en_cliente "
            "no contienen el mismo número de transacciones "
            "para cada combinación t_id + cliente."
        )



    # 7. INCORPORAR LOS DATOS DE VENTA

    # Añadimos sin modificar:
    #
    # - tiempo_venta
    # - precio_venta
    # - peso
    #
    # La combinación:
    #
    #     t_id
    #     cliente
    #     _pos_transaccion
    #
    # identifica cada transacción individual.


    df_relacion = df_relacion.merge(

        df_venta[
            [
                "t_id",
                "cliente",
                "_pos_transaccion",
                "tiempo_venta",
                "precio_venta",
                "peso"
            ]
        ],

        on=[
            "t_id",
            "cliente",
            "_pos_transaccion"
        ],

        how="left",

        validate="one_to_one",

        indicator="_merge_venta"
    )


    if not (
        df_relacion["_merge_venta"] == "both"
    ).all():

        raise ValueError(
            "Hay transacciones de proveedores-clientes "
            "sin correspondencia en venta_en_cliente."
        )


    df_relacion = df_relacion.drop(
        columns=[
            "_merge_venta",
            "_pos_transaccion"
        ]
    )



    # 8. CREAR LA TABLA BASE A PARTIR DE INFO_LOTE

    # info_lote aporta:
    #
    # - t_id
    # - marca
    # - coste_inicial
    # - codigo_lote
    #
    # Para los t_id repetidos utilizamos la posición:
    #
    #     t_id + _pos_tid
    #
    # para vincular cada aparición con la relación comercial
    # construida anteriormente.
    #
    # Éste es precisamente el punto donde aplicamos la hipótesis
    # de correspondencia directa por posición.


    df = df_info.merge(

        df_relacion[
            [
                "t_id",
                "_pos_tid",
                "proveedor",
                "cliente",
                "tiempo_venta",
                "precio_venta",
                "peso"
            ]
        ],

        on=[
            "t_id",
            "_pos_tid"
        ],

        how="left",

        validate="one_to_one",

        indicator="_merge_relacion"
    )


    if not (
        df["_merge_relacion"] == "both"
    ).all():

        raise ValueError(
            "Hay registros de info_lote sin relación "
            "proveedor/cliente/venta."
        )


    df = df.drop(
        columns=[
            "_merge_relacion"
        ]
    )



    # 9. INCORPORAR TIPO Y TIEMPO DE RECOGIDA

    # tipo.json aporta:
    #
    # - tipo
    # - tiempo_recogida
    #
    # Nuevamente:
    #
    #     t_id + _pos_tid
    #
    # relaciona las distintas apariciones de un mismo t_id.
    #
    # No usamos tiempo_recogida para inferir nada.
    # El valor se copia exactamente como viene en el JSON.


    df = df.merge(

        df_tipo[
            [
                "t_id",
                "_pos_tid",
                "tipo",
                "tiempo_recogida"
            ]
        ],

        on=[
            "t_id",
            "_pos_tid"
        ],

        how="left",

        validate="one_to_one",

        indicator="_merge_tipo"
    )


    if not (
        df["_merge_tipo"] == "both"
    ).all():

        raise ValueError(
            "Hay registros de info_lote sin correspondencia "
            "en tipo.json."
        )


    df = df.drop(
        columns=[
            "_merge_tipo"
        ]
    )



    # 10. INCORPORAR SUBTIPO

    # df_subtipos contiene:
    #
    #     t_id
    #     tipo
    #     subtipo
    #
    # En muchos casos:
    #
    #     t_id + tipo
    #
    # ya identifica el subtipo directamente.
    #
    # Pero existen t_id repetidos que pertenecen a varios
    # subtipos del mismo tipo, por ejemplo:
    #
    #     sceneXXXXX.png | Apple | Apple B
    #     sceneXXXXX.png | Apple | Apple D
    #     sceneXXXXX.png | Apple | Apple F
    #
    # Para esos casos utilizamos nuevamente el orden relativo
    # dentro de:
    #
    #     t_id + tipo
    #
    # según la hipótesis posicional acordada.


    df["_pos_tid_tipo"] = (
        df
        .groupby(
            [
                "t_id",
                "tipo"
            ],
            dropna=False
        )
        .cumcount()
    )


    df_sub["_pos_tid_tipo"] = (
        df_sub
        .groupby(
            [
                "t_id",
                "tipo"
            ],
            dropna=False
        )
        .cumcount()
    )


    df = df.merge(

        df_sub[
            [
                "t_id",
                "tipo",
                "_pos_tid_tipo",
                "subtipo"
            ]
        ],

        on=[
            "t_id",
            "tipo",
            "_pos_tid_tipo"
        ],

        how="left",

        validate="one_to_one",

        indicator="_merge_subtipo"
    )


    if not (
        df["_merge_subtipo"] == "both"
    ).all():

        raise ValueError(
            "Hay productos sin correspondencia de subtipo."
        )


    df = df.drop(
        columns=[
            "_merge_subtipo"
        ]
    )



    # 11. COMPROBAR EL CÓDIGO DE LOTE

    # lotes.json contiene:
    #
    #     t_id -> codigo_lote
    #
    # Hemos comprobado que, incluso para los t_id repetidos,
    # todas sus apariciones pertenecen al mismo lote.
    #
    # Por tanto no necesitamos posición:
    #
    #     t_id -> codigo_lote
    #
    # sirve como relación many_to_one.
    #
    # No reemplazamos el codigo_lote de info_lote.
    # Sólo comprobamos que coincide.


    df_lotes_check = df_lotes.rename(
        columns={
            "codigo_lote": "codigo_lote_check"
        }
    )


    df = df.merge(

        df_lotes_check,

        on="t_id",

        how="left",

        validate="many_to_one"
    )


    # Consideramos iguales:
    #
    # valor == valor
    #
    # o
    #
    # ambos valores nulos.
    coincide_lote = (
        df["codigo_lote"].eq(
            df["codigo_lote_check"]
        )
        |
        (
            df["codigo_lote"].isna()
            &
            df["codigo_lote_check"].isna()
        )
    )


    inconsistencias = (
        ~coincide_lote
    ).sum()


    if inconsistencias > 0:

        print(
            f"{inconsistencias} filas con codigo_lote distinto "
            "entre info_lote y lotes.json"
        )

    else:

        print(
            "codigo_lote consistente entre "
            "info_lote y lotes.json"
        )


    df = df.drop(
        columns=[
            "codigo_lote_check"
        ]
    )



    # 12. COMPROBACIÓN FINAL DEL NÚMERO DE FILAS

    # Ningún merge debe:
    #
    # - perder productos;
    # - duplicar productos;
    # - generar productos cartesianos.
    #
    # Debemos terminar exactamente con las mismas filas que
    # info_lote: 70.549 en los datos actuales.


    if len(df) != filas_esperadas:

        raise ValueError(
            f"La construcción ha generado {len(df)} filas; "
            f"se esperaban {filas_esperadas}."
        )



    # 13. ELIMINAR CLAVES TÉCNICAS TEMPORALES

    # Las posiciones sólo han servido para construir las
    # correspondencias.
    #
    # No forman parte del modelo relacional SQL.


    df = df.drop(
        columns=[
            "_pos_tid",
            "_pos_tid_tipo"
        ]
    )



    # 14. ESTRUCTURA FINAL

    # Se conserva EXACTAMENTE la estructura que espera la función:
    #
    #     construccion_tablas_relacionales(df_final)
    #
    # No hay que modificar las celdas posteriores.


    columnas_orden = [
        "t_id",
        "tipo",
        "subtipo",
        "marca",
        "codigo_lote",
        "proveedor",
        "cliente",
        "coste_inicial",
        "tiempo_recogida",
        "precio_venta",
        "tiempo_venta",
        "peso",
    ]


    df = df[
        columnas_orden
    ].copy()


    return df

In [44]:
df_final = construccion_df_final(datos, df_subtipos)
print(f"Filas: {len(df_final)}")
print(f"Columnas: {list(df_final.columns)}")
df_final.head()

codigo_lote consistente entre info_lote y lotes.json
Filas: 70549
Columnas: ['t_id', 'tipo', 'subtipo', 'marca', 'codigo_lote', 'proveedor', 'cliente', 'coste_inicial', 'tiempo_recogida', 'precio_venta', 'tiempo_venta', 'peso']


,t_id,tipo,subtipo,marca,codigo_lote,proveedor,cliente,coste_inicial,tiempo_recogida,precio_venta,tiempo_venta,peso
0,Apple 1.png,Apple,Apple A,ParaísoFrutal,G80V76K49J80T71,Agricultura Inteligente TechCultivos,CompraMaestra,2.642048,433,4.909680,437.0,264.195357
1,Apple 10.png,Apple,Apple A,TropicalSabor,G80V76K49L46V78M,Semillero Genético BioCampo,CompraMaestra,1.702296,252,3.425103,259.0,141.647327
2,Apple 100.png,Apple,Apple A,DeliciaNaturaleza,G80V76K49L48J80T71,Pesca Sustentable Oceanica,La Tienda Justa,1.176703,442,3.178386,447.0,151.923586
3,Apple 101.png,Apple,Apple A,EmbrujoFrutal,G80V76K49L49J80T71,Carnes Sostenibles CampoReal,Supermercados del Valle,2.314693,647,4.126968,658.0,442.475391
4,Apple 102.png,Apple,Apple A,ExóticoManjar,G80V76K49L50J80T71,Silos y Almacenes AgroVault,Tienda Familiar,2.473541,418,3.349878,428.0,423.776864


In [45]:
# 1. ¿El número de filas es exactamente 70.549?
assert len(df_final) == 70549, f"Esperaba 70549 filas, obtuve {len(df_final)}"

# 2. ¿Hay alguna fila con NaN en columnas críticas?
print("\nNulos por columna:")
print(df_final.isna().sum())

# 3. ¿Cómo queda nuestro scene de prueba?
print("\n=== scene00201.png ===")
print(df_final[df_final["t_id"] == "scene00201.png"])


Nulos por columna:
t_id                  0
tipo                  0
subtipo               0
marca                 0
codigo_lote           0
proveedor             0
cliente               0
coste_inicial      2011
tiempo_recogida       0
precio_venta        694
tiempo_venta        223
peso                  0
dtype: int64

=== scene00201.png ===
                t_id   tipo  subtipo             marca            codigo_lote                             proveedor                 cliente  coste_inicial  tiempo_recogida  precio_venta  \
1222  scene00201.png  Apple  Apple B   VibranteSabores  C67K78K48L50L49J80T71  Agricultura Inteligente TechCultivos      Alimentación Total       1.474485              175      3.123456   
1223  scene00201.png  Apple  Apple D     DeliciosoEdén  C67K78K48L50L49J80T71          Carnes Sostenibles CampoReal  Distribuciones del Sol       1.924915              346      3.543184   
1224  scene00201.png  Apple  Apple F     PaladarDorado  C67K78K48L50L49J80T71           

In [46]:
# 1. ¿Coinciden los nulos de tiempo_venta y precio_venta?
ambos_null = df_final[df_final["tiempo_venta"].isna() & df_final["precio_venta"].isna()]
solo_tv_null = df_final[df_final["tiempo_venta"].isna() & df_final["precio_venta"].notna()]
solo_pv_null = df_final[df_final["tiempo_venta"].notna() & df_final["precio_venta"].isna()]

print(f"Ambos nulos (tiempo_venta y precio_venta): {len(ambos_null)}")
print(f"Solo tiempo_venta nulo: {len(solo_tv_null)}")
print(f"Solo precio_venta nulo: {len(solo_pv_null)}")

# 2. ¿Los nulos están concentrados en los scene o también en productos normales?
df_final["es_scene"] = df_final["t_id"].str.startswith("scene")
print("\nNulos en scene vs no-scene:")
print(df_final.groupby("es_scene")[["coste_inicial", "precio_venta", "tiempo_venta"]].apply(lambda x: x.isna().sum()))

# 3. ¿Hay coste_inicial nulo en productos que SÍ se vendieron?
sin_coste_pero_vendidos = df_final[df_final["coste_inicial"].isna() & df_final["precio_venta"].notna()]
print(f"\nProductos vendidos sin coste registrado: {len(sin_coste_pero_vendidos)}")

df_final = df_final.drop(columns=["es_scene"])

Ambos nulos (tiempo_venta y precio_venta): 2
Solo tiempo_venta nulo: 221
Solo precio_venta nulo: 692

Nulos en scene vs no-scene:
          coste_inicial  precio_venta  tiempo_venta
es_scene                                           
False              1957           671           217
True                 54            23             6

Productos vendidos sin coste registrado: 1989


Se decide mantener los valores nulos de tiempo venta y precio venta en la tabla de ventas para un posterior analisis e identificación de errores. Ya que estos datos no necesariamente son errores, pueden ser: transacciones en curso, ventas a precio 0, errores o frutas dañadas en la manipulación de entrega al cliente final.

### 2º) Función para dividir los DF acorde a la BBDD relacional.

Se incluye la conversión de los datos 'tiempo_recogida' y 'tiempo_venta' a u formato datetime, para una mejor visualización.

In [47]:
def construccion_tablas_relacionales(df_final):
    df = df_final.copy()
    origen = pd.Timestamp("2025-09-01 07:00:00")
    df["tiempo_recogida"] = origen + pd.to_timedelta(df["tiempo_recogida"], unit="h")
    df["tiempo_venta"] = origen + pd.to_timedelta(df["tiempo_venta"], unit="h")

    def construir_catalogo(serie, id_col, nombre_col):
        unicos = sorted(serie.dropna().unique())
        return pd.DataFrame({
            id_col: range(1, len(unicos) + 1),
            nombre_col: unicos,
        })

    df_tipo = construir_catalogo(df["tipo"], "id_tipo", "nombre")
    df_marca = construir_catalogo(df["marca"], "id_marca", "nombre")
    df_proveedor = construir_catalogo(df["proveedor"], "id_proveedor", "nombre")
    df_cliente = construir_catalogo(df["cliente"], "id_cliente", "nombre")
    df_lote = construir_catalogo(df["codigo_lote"], "id_lote", "codigo")

    pares_subtipo = (
        df[["subtipo", "tipo"]]
        .drop_duplicates()
        .sort_values(["tipo", "subtipo"])
        .reset_index(drop=True)
    )
    pares_subtipo["id_subtipo"] = range(1, len(pares_subtipo) + 1)

    pares_subtipo = pares_subtipo.merge(df_tipo, left_on="tipo", right_on="nombre", how="left")
    df_subtipo = pares_subtipo[["id_subtipo", "subtipo", "id_tipo"]].rename(columns={"subtipo": "nombre"})

    df_producto = df.copy()

    df_producto = df_producto.merge(df_tipo.rename(columns={"nombre": "tipo"}), on="tipo", how="left")
    df_producto = df_producto.merge(
        df_subtipo.rename(columns={"nombre": "subtipo"})[["id_subtipo", "subtipo", "id_tipo"]],
        on=["subtipo", "id_tipo"], how="left"
    )
    df_producto = df_producto.merge(df_marca.rename(columns={"nombre": "marca"}), on="marca", how="left")
    df_producto = df_producto.merge(df_proveedor.rename(columns={"nombre": "proveedor"}), on="proveedor", how="left")
    df_producto = df_producto.merge(df_lote.rename(columns={"codigo": "codigo_lote"}), on="codigo_lote", how="left")

    df_producto = df_producto.reset_index(drop=True)
    df_producto["id_producto"] = range(1, len(df_producto) + 1)

    df_producto_final = df_producto[[
        "id_producto", "t_id", "id_tipo", "id_subtipo", "id_marca",
        "id_proveedor", "id_lote", "coste_inicial", "tiempo_recogida"
    ]]

    df_venta = df_producto.merge(
        df_cliente.rename(columns={"nombre": "cliente"}), on="cliente", how="left"
    )
    df_venta = df_venta.reset_index(drop=True)
    df_venta["id_venta"] = range(1, len(df_venta) + 1)

    df_venta_final = df_venta[[
        "id_venta", "id_producto", "id_cliente",
        "precio_venta", "tiempo_venta", "peso"
    ]]

    return {
        "tipo": df_tipo,
        "subtipo": df_subtipo,
        "marca": df_marca,
        "proveedor": df_proveedor,
        "cliente": df_cliente,
        "lote": df_lote,
        "producto": df_producto_final,
        "venta": df_venta_final,
    }

In [48]:
tablas = construccion_tablas_relacionales(df_final)

for nombre, df in tablas.items():
    print(f"{nombre:12s} {df.shape}")

tipo         (15, 2)
subtipo      (27, 3)
marca        (35, 2)
proveedor    (35, 2)
cliente      (34, 2)
lote         (69567, 2)
producto     (70549, 9)
venta        (70549, 6)


In [49]:
# 1. IDs consecutivos
for nombre in ["tipo", "subtipo", "marca", "proveedor", "cliente", "lote", "producto", "venta"]:
    df = tablas[nombre]
    id_col = [c for c in df.columns if c.startswith("id_")][0]
    print(f"{nombre:12s} {id_col}: min={df[id_col].min()}, max={df[id_col].max()}, count={len(df)}")

# 2. FKs nulas
print("\nFKs nulas en producto:")
print(tablas["producto"][["id_tipo", "id_subtipo", "id_marca", "id_proveedor", "id_lote"]].isna().sum())

print("\nFKs nulas en venta:")
print(tablas["venta"][["id_producto", "id_cliente"]].isna().sum())

tipo         id_tipo: min=1, max=15, count=15
subtipo      id_subtipo: min=1, max=27, count=27
marca        id_marca: min=1, max=35, count=35
proveedor    id_proveedor: min=1, max=35, count=35
cliente      id_cliente: min=1, max=34, count=34
lote         id_lote: min=1, max=69567, count=69567
producto     id_producto: min=1, max=70549, count=70549
venta        id_venta: min=1, max=70549, count=70549

FKs nulas en producto:
id_tipo         0
id_subtipo      0
id_marca        0
id_proveedor    0
id_lote         0
dtype: int64

FKs nulas en venta:
id_producto    0
id_cliente     0
dtype: int64


# 3. Descarga los CSV para carga posterior en SQL

In [52]:
# ============================================================
# EXPORTAR LAS TABLAS RELACIONALES A CSV
# ============================================================

from pathlib import Path
from google.colab import files
import shutil


# ============================================================
# 1. PEDIR NOMBRE DE LA CARPETA DE SALIDA
# ============================================================

nombre_carpeta = input(
    "Introduce el nombre de la carpeta para exportar los CSV "
    "[Enter = vision_artificial_csv]: "
).strip()

if nombre_carpeta == "":
    nombre_carpeta = "vision_artificial_csv"


# Evitamos caracteres problemáticos en el nombre de la carpeta
nombre_carpeta = (
    nombre_carpeta
    .replace("/", "_")
    .replace("\\", "_")
    .replace(":", "_")
)


# ============================================================
# 2. CREAR CARPETA DE EXPORTACIÓN EN COLAB
# ============================================================

ruta_salida = Path("/content") / nombre_carpeta

ruta_salida.mkdir(
    parents=True,
    exist_ok=True
)

print(f"\nCarpeta de exportación: {ruta_salida}")


# ============================================================
# 3. ORDEN DE EXPORTACIÓN
# ============================================================
#
# Conservamos el mismo orden lógico que se iba a utilizar
# para cargar las tablas en MySQL.
# ============================================================

orden_exportacion = [
    "tipo",
    "marca",
    "proveedor",
    "cliente",
    "lote",
    "subtipo",
    "producto",
    "venta",
]


# ============================================================
# 4. EXPORTAR CADA DATAFRAME A CSV
# ============================================================

print("\nExportando tablas...\n")

for nombre in orden_exportacion:

    df = tablas[nombre]

    ruta_csv = ruta_salida / f"{nombre}.csv"

    df.to_csv(
        ruta_csv,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"✓ {nombre:12s} "
        f"{len(df):>7,} filas "
        f"→ {ruta_csv.name}"
    )


# ============================================================
# 5. COMPROBAR LOS ARCHIVOS GENERADOS
# ============================================================

archivos_generados = list(
    ruta_salida.glob("*.csv")
)

if len(archivos_generados) != len(orden_exportacion):

    raise RuntimeError(
        f"Se esperaban {len(orden_exportacion)} CSV "
        f"pero se han generado {len(archivos_generados)}."
    )


# ============================================================
# 6. COMPRIMIR LA CARPETA
# ============================================================

ruta_zip_base = Path("/content") / nombre_carpeta

ruta_zip = shutil.make_archive(
    base_name=str(ruta_zip_base),
    format="zip",
    root_dir=ruta_salida.parent,
    base_dir=ruta_salida.name
)


# ============================================================
# 7. RESUMEN
# ============================================================

print("\n-----------------------------------")
print("EXPORTACIÓN COMPLETADA")
print("-----------------------------------")
print(f"Carpeta: {ruta_salida}")
print(f"CSV generados: {len(archivos_generados)}")
print(f"ZIP: {ruta_zip}")

print("\nContenido:")

for archivo in sorted(archivos_generados):
    print(f" ✓ {archivo.name}")


# ============================================================
# 8. DESCARGAR EL ZIP
# ============================================================

print("\nIniciando descarga...")

files.download(ruta_zip)

Introduce el nombre de la carpeta para exportar los CSV [Enter = vision_artificial_csv]: 

Carpeta de exportación: /content/vision_artificial_csv

Exportando tablas...

✓ tipo              15 filas → tipo.csv
✓ marca             35 filas → marca.csv
✓ proveedor         35 filas → proveedor.csv
✓ cliente           34 filas → cliente.csv
✓ lote          69,567 filas → lote.csv
✓ subtipo           27 filas → subtipo.csv
✓ producto      70,549 filas → producto.csv
✓ venta         70,549 filas → venta.csv

-----------------------------------
EXPORTACIÓN COMPLETADA
-----------------------------------
Carpeta: /content/vision_artificial_csv
CSV generados: 8
ZIP: /content/vision_artificial_csv.zip

Contenido:
 ✓ cliente.csv
 ✓ lote.csv
 ✓ marca.csv
 ✓ producto.csv
 ✓ proveedor.csv
 ✓ subtipo.csv
 ✓ tipo.csv
 ✓ venta.csv

Iniciando descarga...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>